In [8]:
from Bio import Entrez
import requests

def pubmed_count_requests(drug, disease):
    term = f'"{drug}"[Title/Abstract] AND "{disease}"[Title/Abstract]'
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": term,
        "retmode": "json",
        "email": "oseicharlotte633@gmail.com" # Replace with your email
    }

    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return int(r.json()["esearchresult"]["count"])

count_1 = pubmed_count_requests("dexamethasone", "")
count_2 = pubmed_count_requests("dexamethasone", "Depression")
count_3 = pubmed_count_requests("dexamethasone", "Hypertension")
count_4 = pubmed_count_requests("dexamethasone", "Rheumatoid Arthritis")

KeyError: 'count'

In [20]:
print(f"Number of PubMed articles for dexamethasone and Asthma: {count_1}")
print(f"Number of PubMed articles for dexamethasone and Depression: {count_2}")
print(f"Number of PubMed articles for dexamethasone and Hypertension: {count_3}")
print(f"Number of PubMed articles for dexamethasone and Rheumatoid Arthritis: {count_4}")

Number of PubMed articles for dexamethasone and Asthma: 70190
Number of PubMed articles for dexamethasone and Depression: 2018
Number of PubMed articles for dexamethasone and Hypertension: 1846
Number of PubMed articles for dexamethasone and Rheumatoid Arthritis: 533


In [ ]:
import requests
import json
from xml.etree import ElementTree as ET
from time import sleep

def pmc_get_pmids(drug, disease, max_count=20):
    term = f'"{drug}"[Title/Abstract] AND "{disease}"[Title/Abstract]'
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pmc",
        "term": term,
        "retmode": "json",
        "retmax": max_count,
        "email": "oseifrancis633@gmail.com"
    }

    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()
    return data.get("esearchresult", {}).get("idlist", [])

def pmc_fetch_abstracts(pmids):
    articles = []
    for i in range(0, len(pmids), 20):
        batch = pmids[i:i+20]
        url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        params = {
            "db": "pmc",
            "id": ",".join(batch),
            "retmode": "xml",
            "email": "oseifrancis633@gmail.com"
        }

        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        root = ET.fromstring(r.content)

        for article in root.findall(".//article"):
            title_elem = article.find(".//article-title")
            abstract_elem = article.find(".//abstract")

            title = "".join(title_elem.itertext()).strip() if title_elem is not None else "No Title"
            abstract = "".join(abstract_elem.itertext()).strip() if abstract_elem is not None else "No Abstract"
            
            articles.append({
                "title": title,
                "abstract": abstract
            })

        sleep(0.5)

    return articles

# Example usage
pmids = pmc_get_pmids("dexamethasone", "Depression", max_count=10)
results_json = pmc_fetch_abstracts(pmids)

# Output JSON to screen
print(json.dumps(results_json, indent=2))


[
  {
    "title": "Cortisol and the Dexamethasone Suppression Test as a Biomarker for Melancholic Depression: A Narrative Review",
    "abstract": "The dexamethasone suppression test (DST) assesses the functionality of the HPA axis and can be regarded as the first potential biomarker in psychiatry. In 1981, a group of researchers at the University of Michigan published a groundbreaking paper regarding its use for diagnosing melancholic depression, reporting a diagnostic sensitivity of 67% and a specificity of 95%. While this study generated much enthusiasm and high expectations in the field of biological psychiatry, subsequent studies produced equivocal results, leading to the test being rejected by the American Psychiatric Association. The scientific reasons leading to the rise and fall of the DST are assessed in this review, suggestions are provided as to how the original test can be improved, and its potential applications in clinical psychiatry are discussed. An improved, standard

In [1]:
import networkx as nx

# Load the graph
G = nx.read_graphml("graph/bipartite.graphml")

# Identify node types
drug_nodes = {n for n, d in G.nodes(data=True) if d.get("type") == "drug"}
disease_nodes = {n for n, d in G.nodes(data=True) if d.get("type") == "disease"}

# Count edges
num_edges = G.number_of_edges()

# Optional: Identify and skip placebo entries (assuming a node or edge attribute marks them)
placebo_skipped = sum(
    1 for n, d in G.nodes(data=True)
    if "placebo" in d.get("name", "").lower() or "placebo" in str(n).lower()
)

# Print stats
print(f"Drug nodes: {len(drug_nodes)}")
print(f"Disease nodes: {len(disease_nodes)}")
print(f"Edges: {num_edges}")
print(f"Placebo entries skipped: {placebo_skipped}")


Drug nodes: 441
Disease nodes: 353
Edges: 1200
Placebo entries skipped: 0


In [4]:
import pandas as pd
import networkx as nx

# Load drug centralities data
df = pd.read_csv("graph/drug_centrality.csv")

# Load the bipartite graph
G = nx.read_graphml("graph/bipartite.graphml")

# Extract drug nodes only
drug_nodes = {n for n, d in G.nodes(data=True) if d.get("type") == "drug"}

# Compute PageRank scores
pagerank_scores = nx.pagerank(G)
pagerank_drug_scores = {
    node: score for node, score in pagerank_scores.items() if node in drug_nodes
}

# Prepare PageRank DataFrame
pagerank_df = pd.DataFrame([
    {"Drug": node, "PageRank": score}
    for node, score in pagerank_drug_scores.items()
])

# Merge PageRank with centrality DataFrame (left join on drug name)
df_combined = df.merge(pagerank_df, on="Drug", how="left")

# Top 5 drugs by each centrality metric
top_eigen = df_combined.sort_values(by="EigenvectorCentrality", ascending=False).head(5)
top_pagerank = df_combined.sort_values(by="PageRank", ascending=False).head(5)
top_degree = df_combined.sort_values(by="DegreeCentrality", ascending=False).head(5)
top_betweenness = df_combined.sort_values(by="BetweennessCentrality", ascending=False).head(5)

# Display results
print("Top 5 Drugs by Eigenvector Centrality:")
print(top_eigen[["Drug", "EigenvectorCentrality"]], end="\n\n")

print("Top 5 Drugs by PageRank:")
print(top_pagerank[["Drug", "PageRank"]], end="\n\n")

print("Top 5 Drugs by Degree Centrality:")
print(top_degree[["Drug", "DegreeCentrality"]], end="\n\n")

print("Top 5 Drugs by Betweenness Centrality:")
print(top_betweenness[["Drug", "BetweennessCentrality"]], end="\n\n")


Top 5 Drugs by Eigenvector Centrality:
              Drug  EigenvectorCentrality
2    rosiglitazone               0.169261
54       ritonavir               0.158384
39        rifampin               0.156791
364     leucovorin               0.150467
309    alendronate               0.147252

Top 5 Drugs by PageRank:
                 Drug  PageRank
255  cyclophosphamide  0.008208
357     dexamethasone  0.006751
51          metformin  0.005653
427   saline solution  0.004542
203       epinephrine  0.004534

Top 5 Drugs by Degree Centrality:
              Drug  DegreeCentrality
2    rosiglitazone          0.320455
364     leucovorin          0.243182
105   atorvastatin          0.227273
40      prednisone          0.218182
54       ritonavir          0.218182

Top 5 Drugs by Betweenness Centrality:
                 Drug  BetweennessCentrality
2       rosiglitazone               0.114096
255  cyclophosphamide               0.061357
63       prednisolone               0.058467
51          me

In [1]:
import os
import json

def check_nct_ids(data_dir="data"):
    all_ids = []
    unique_ids = set()

    # Get all .json files in the data directory
    json_files = [f for f in os.listdir(data_dir) if f.endswith(".json")]

    print(f"Checking {len(json_files)} file(s) in '{data_dir}'...")

    for file in json_files:
        file_path = os.path.join(data_dir, file)
        with open(file_path, "r", encoding="utf-8") as f:
            trials = json.load(f)
            for trial in trials:
                nct_id = trial.get("nctId")
                if nct_id:
                    all_ids.append(nct_id)
                    unique_ids.add(nct_id)

    print(f"\nTotal NCT IDs (with duplicates): {len(all_ids)}")
    print(f"Unique NCT IDs: {len(unique_ids)}")

    duplicates = len(all_ids) - len(unique_ids)
    print(f"Duplicate NCT IDs found: {duplicates}")

if __name__ == "__main__":
    check_nct_ids("data")


Checking 10 file(s) in 'data'...

Total NCT IDs (with duplicates): 9808
Unique NCT IDs: 9808
Duplicate NCT IDs found: 0


In [32]:
import requests

class DrugSideEffects:
    def __init__(self):
        self.api_url = "https://api.fda.gov/drug/event.json"
    
    def get_side_effects(self, drug_name):
        """Get only side effects for specified drug"""
        params = {
            'search': f'patient.drug.medicinalproduct:"{drug_name.lower()}"',
            'limit': 5  # Get more cases for better coverage
        }
        
        try:
            response = requests.get(self.api_url, params=params)
            response.raise_for_status()
            data = response.json()
            
            side_effects = set()  # Using set to avoid duplicates
            
            if data.get('results'):
                for case in data['results']:
                    for reaction in case.get('patient', {}).get('reaction', []):
                        effect = reaction.get('reactionmeddrapt')
                        if effect:
                            side_effects.add(effect.capitalize())
                
                return {
                    'drug': drug_name.capitalize(),
                    'side_effects': sorted(side_effects)  # Return alphabetized list
                }
            
            return {
                'drug': drug_name.capitalize(),
                'side_effects': ["No side effects data found"]
            }
            
        except requests.exceptions.RequestException:
            return {
                'drug': drug_name.capitalize(),
                'side_effects': ["Error accessing FDA database"]
            }

# Example usage:
searcher = DrugSideEffects()
print(searcher.get_side_effects("Sirolimus"))  # Just change this drug name

{'drug': 'Sirolimus', 'side_effects': ['Angina pectoris', 'Dehydration', 'Drug interaction', 'Drug-induced liver injury', 'Heart disease congenital', 'Immunosuppressant drug level increased', 'Infection', 'Inflammation', 'Low birth weight baby', 'Malaise', 'Metastatic malignant melanoma', 'Nephrogenic systemic fibrosis', 'Oral surgery', 'Pain', 'Pulmonary artery atresia', 'Purulent discharge', 'Seborrhoeic keratosis', 'Skin lesion', 'Ulcer haemorrhage', 'Viith nerve paralysis']}


In [ ]:
import json
from collections import defaultdict

# Load the JSON file
with open('processed_data/unmatched_pairs.json', 'r') as f:
    data = json.load(f)

# Initialize counters
condition_stats = defaultdict(lambda: {'unmatched_condition': 0, 'unmatched_intervention': 0})
unique_conditions = set()

# Process each entry
for entry in data:
    condition = entry['condition']
    unique_conditions.add(condition)
    
    if entry['reason'] == 'unmatched condition':
        condition_stats[condition]['unmatched_condition'] += 1
    elif entry['reason'] == 'unmatched intervention':
        condition_stats[condition]['unmatched_intervention'] += 1

# Calculate totals
total_unique_conditions = len(unique_conditions)
total_unmatched_condition = sum(stats['unmatched_condition'] for stats in condition_stats.values())
total_unmatched_intervention = sum(stats['unmatched_intervention'] for stats in condition_stats.values())

# Print results
print(f"Total unique conditions: {total_unique_conditions}")
print(f"Conditions with 'unmatched condition': {total_unmatched_condition}")
print(f"Conditions with 'unmatched intervention': {total_unmatched_intervention}")

21057

In [35]:
import json
from collections import defaultdict

# Load the JSON file
with open('processed_data/condition_drug_pairs.json', 'r') as f:
    data = json.load(f)

# Initialize sets to store unique values
unique_conditions = set()
unique_interventions = set()
unique_phases = set()
unique_statuses = set()

# Process each entry
for entry in data:
    unique_conditions.add(entry['condition'])
    
    if entry['intervention'] is not None:
        unique_interventions.add(entry['intervention'])
    
    if 'phases' in entry and entry['phases'] is not None:
        for phase in entry['phases']:
            unique_phases.add(phase)
    
    if 'status' in entry and entry['status'] is not None:
        unique_statuses.add(entry['status'])

# Calculate counts
total_rows = len(data)
total_unique_conditions = len(unique_conditions)
total_unique_interventions = len(unique_interventions)
total_unique_phases = len(unique_phases)
total_unique_statuses = len(unique_statuses)

# Print results
print(f"Total rows: {total_rows}")
print(f"Unique conditions: {total_unique_conditions}")
print(f"Unique interventions: {total_unique_interventions}")
print(f"Unique phases: {total_unique_phases}")
print(f"Unique statuses: {total_unique_statuses}")

Total rows: 10178
Unique conditions: 847
Unique interventions: 873
Unique phases: 4
Unique statuses: 6


In [52]:
import requests
import re
import time
import json
import os
from collections import defaultdict

# Create output folder if not exists
os.makedirs("model_evaluation", exist_ok=True)

# Cancer diseases
diseases_kidney_diseases = [
                    "Chronic Kidney Disease",           
                    "Acute Kidney Injury",             
                    "renal insufficiency, chronic",         
                    "Polycystic Kidney Diseases",              
                    "Nephrotic Syndrome",              
                    "Glomerulonephritis",              
                    "glomerulonephritis, iga",                   
                    "glomerulonephritis, membranous",         
                    "carcinoma, renal cell"             
                ]


# Valid statuses
VALID_STATUSES = {"COMPLETED", "RECRUITING", "ACTIVE_NOT_RECRUITING", "ENROLLING_BY_INVITATION","TERMINATED"}

# Regex to extract phase from title
PHASE_PATTERN = re.compile(r'(PHASE\s?[I1]{1,3}|PHASE\s?[1-4])', re.IGNORECASE)

# API endpoint
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

# Extract phase from trial title
def extract_phase(title):
    match = PHASE_PATTERN.search(title)
    return match.group(0).upper().replace(" ", "") if match else None

# Query trials for drug and disease
def get_trials(drug, disease, max_trials=100):
    params = {
        "query.term": f"{drug} AND {disease}",
        "pageSize": max_trials
    }
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json().get("studies", [])

# Storage
trial_json_data = {}
phase_status_counts = {}

# Main loop
drug_name = "metformin"

for disease in diseases_kidney_diseases:
    trials = get_trials(drug_name, disease)
    phase_status = defaultdict(lambda: defaultdict(int))
    trial_records = []

    for trial in trials:
        section = trial.get("protocolSection", {})
        title = section.get("identificationModule", {}).get("officialTitle", "")
        status = section.get("statusModule", {}).get("overallStatus", "").upper()
        nct_id = section.get("identificationModule", {}).get("nctId", "")
        phase = extract_phase(title)

        if status in VALID_STATUSES and phase:
            phase_status[phase][status] += 1
            trial_records.append({
                "nct_id": nct_id,
                "title": title,
                "status": status,
                "phase": phase,
                "disease": disease,
                "drug": drug_name
            })

    print(f"\n==== {disease} ====")
    for ph in sorted(phase_status.keys()):
        print(f"{ph}")
        for stat in sorted(phase_status[ph].keys()):
            print(f"  - {stat}: {phase_status[ph][stat]}")

    phase_status_counts[disease] = phase_status
    trial_json_data[disease] = trial_records
    time.sleep(1.5)

# Save results using f-strings for correct filenames
with open(f"model_evaluation/trial_metadata_diseases_kidney_{drug_name}.json", "w") as f:
    json.dump(trial_json_data, f, indent=2)

with open(f"model_evaluation/phase_status_counts_by_diseases_kidney_{drug_name}.json", "w") as f:
    json.dump(phase_status_counts, f, indent=2)

print("\n Saved outputs to 'model_evaluation/' folder:")
print(f"- trial_metadata_cancer_{drug_name}.json")
print(f"- phase_status_counts_by_cancer_{drug_name}.json")



==== Chronic Kidney Disease ====
PHASE3
  - RECRUITING: 1
PHASEIII
  - COMPLETED: 1

==== Acute Kidney Injury ====

==== renal insufficiency, chronic ====
PHASE3
  - RECRUITING: 1
PHASEIII
  - COMPLETED: 1

==== Polycystic Kidney Diseases ====

==== Nephrotic Syndrome ====

==== Glomerulonephritis ====

==== glomerulonephritis, iga ====

==== glomerulonephritis, membranous ====

==== carcinoma, renal cell ====
PHASEI
  - COMPLETED: 1
  - TERMINATED: 1
PHASEII
  - RECRUITING: 1
  - TERMINATED: 1

 Saved outputs to 'model_evaluation/' folder:
- trial_metadata_cancer_metformin.json
- phase_status_counts_by_cancer_metformin.json
